# Full_full LoRaX site-level MIL / attention screen

Plot-only notebook for mixed-granularity molecule-site GIN × mean protein ESM-1b experiments.

Data protocol:

- transductive: genuine LoRaX `rand_split_1..5`;
- inductive_molecule: cold-molecule split seeds `42..46` from the fold-1 full pair pool;
- molecule keys: LoRaX SMILES → InChIKey via `data/processed/molecules/lorax_smiles_to_inchikey.csv`;
- molecule embeddings: `gin_supervised_contextpred_all_m2or[_per_atom].npz`;
- protein embeddings: LoRaX `esm1b_650m_mean_lorax.npz`.

Run script:

```bash
bash scripts/queues/run_full_full_site_mil_attention.sh
```

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t as student_t

sns.set_theme(style='whitegrid', context='notebook')
ROOT = Path('..').resolve().parent if Path.cwd().name == 'baselines_research' else Path.cwd()
RESULT_DIR = ROOT / 'results' / 'attention' / 'full_full' / 'site_mil'
METRICS = ['AUROC', 'AUPRC', 'MCC', 'F1']
metrics_path = RESULT_DIR / 'metrics.csv'
print('results:', RESULT_DIR)
print('metrics exists:', metrics_path.exists())

In [ ]:
if metrics_path.exists():
    runs = pd.read_csv(metrics_path)
else:
    runs = pd.DataFrame(columns=['regime', 'repeat', 'model', *METRICS])

display_cols = ['regime', 'repeat', 'model', 'threshold', 'n_train', 'n_val', 'n_test', 'test_pos', *METRICS]
available_cols = [c for c in display_cols if c in runs.columns]
print(f'loaded {len(runs)} runs')
if len(runs):
    display(runs[available_cols].sort_values(['regime', 'model', 'repeat']).head(20))
else:
    print('No runs yet. Launch scripts/queues/run_full_full_site_mil_attention.sh on the server.')

In [ ]:
def ci95(x):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x)
    if n == 0:
        return pd.Series({'mean': np.nan, 'sd': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n': 0})
    mean = float(x.mean())
    sd = float(x.std(ddof=1)) if n > 1 else np.nan
    half = float(student_t.ppf(0.975, n - 1) * sd / math.sqrt(n)) if n > 1 else np.nan
    return pd.Series({'mean': mean, 'sd': sd, 'ci_low': mean - half if n > 1 else np.nan, 'ci_high': mean + half if n > 1 else np.nan, 'n': n})

if len(runs):
    summary = (
        runs.groupby(['regime', 'model'])[METRICS]
        .agg(lambda s: ci95(s)['mean'])
        .reset_index()
        .sort_values(['regime', 'AUPRC'], ascending=[True, False])
    )
    ci_rows = []
    for (regime, model), g in runs.groupby(['regime', 'model']):
        for metric in METRICS:
            row = ci95(g[metric]).to_dict()
            row.update({'regime': regime, 'model': model, 'metric': metric})
            ci_rows.append(row)
    ci = pd.DataFrame(ci_rows)
    display(summary)
else:
    summary = pd.DataFrame()
    ci = pd.DataFrame()

In [ ]:
if len(ci):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=False)
    axes = axes.ravel()
    for ax, metric in zip(axes, METRICS):
        d = ci[ci.metric == metric].copy()
        d['err_low'] = d['mean'] - d['ci_low']
        d['err_high'] = d['ci_high'] - d['mean']
        sns.barplot(data=d, x='model', y='mean', hue='regime', ax=ax, errorbar=None)
        for container in ax.containers:
            ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
        ax.set_title(metric)
        ax.set_xlabel('')
        ax.set_ylabel(metric)
        ax.tick_params(axis='x', rotation=35)
        ax.legend(title='regime')
    fig.suptitle('Full_full LoRaX site-MIL/attention screen: mean over repeats', y=1.02)
    plt.tight_layout()
else:
    print('No CI plot yet.')

In [ ]:
if len(runs):
    coverage_cols = [c for c in runs.columns if c.endswith('_rows_after') or c in ['regime', 'repeat', 'model', 'n_train', 'n_val', 'n_test']]
    coverage = runs[coverage_cols].drop_duplicates(['regime', 'repeat']).sort_values(['regime', 'repeat'])
    display(coverage)
else:
    print('No coverage table yet.')

In [ ]:
pred_dir = RESULT_DIR / 'predictions'
if len(runs) and pred_dir.exists():
    best = runs.sort_values(['regime', 'AUPRC'], ascending=[True, False]).groupby('regime').head(1)
    pred_frames = []
    for row in best.itertuples():
        run_id = row.run_id if hasattr(row, 'run_id') else f'{row.regime}_rep{row.repeat}_{row.model}_seed{row.seed}_boost{row.boost_seed}'
        path = pred_dir / f'test_{run_id}.csv'
        if path.exists():
            df = pd.read_csv(path)
            df['regime'] = row.regime
            df['model'] = row.model
            df['repeat'] = row.repeat
            pred_frames.append(df)
    if pred_frames:
        pred = pd.concat(pred_frames, ignore_index=True)
        g = sns.displot(data=pred, x='p', hue='y', col='regime', row='model', kind='hist', bins=40, stat='density', common_norm=False, facet_kws={'sharex': True, 'sharey': False}, height=3.0, aspect=1.5)
        g.fig.suptitle('Best-by-AUPRC test score distributions', y=1.03)
    else:
        print('Prediction files for best runs were not found yet.')
else:
    print('No predictions yet.')